## Install

In [ ]:
%pip install --upgrade -q ipython
%pip install -q chromadb
%pip install -q langchain-groq

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Make src/simple_rag_chatbot importable
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

from simple_rag_chatbot.config import Config

config = Config()
print("Groq model:", config.groq_model)
print("Chroma path:", config.chroma_path)

## 2. Ingest documents

Put a few `.pdf`, `.txt`, or `.md` files in `data/sample_docs/` and run the cell below. This chunks each file, embeds the chunks with a sentence-transformer, and upserts them into a persistent Chroma collection. Re-running is safe because chunk IDs are content hashes.

In [ ]:
from simple_rag_chatbot.ingest import load_documents_into_chroma

load_documents_into_chroma(
    docs_dir=REPO_ROOT / "data" / "sample_docs",
    chroma_path=config.chroma_path,
    collection_name=config.collection,
    embed_model_name=config.embed_model,
)

## 3. Build the session

`build_session` loads the embedding and reranking models, connects to Chroma, builds the LangChain chains, generates document-level metadata (cached after the first run), and builds the multi-vector indexes. This is the same function the CLI (`simple-rag-chatbot CLI`) uses.

In [ ]:
from simple_rag_chatbot.session import build_session

session = build_session(config, run_metadata_enrichment=False)

In [ ]:
print("Sources:", session.all_sources)
print()
print("Available metadata fields:", list(session.available_metadata.keys()))

In [ ]:
from simple_rag_chatbot.metadata_enrichment import get_document_metadata

for source, meta in get_document_metadata(session.collection).items():
    print(source)
    print("  summary:", meta.get("summary"))
    print("  category:", meta.get("document_category"))
    print("  entities:", meta.get("entities"))
    print()

## 4. Try each pipeline stage on its own

Before running the full graph, call each chain/function directly so you can inspect exactly what it returns.

### 4a. Intent classification

In [ ]:
question = "How is employee performance evaluated?"

intent_result = session.chains["intent_chain"].invoke({
    "history": "No previous conversation.",
    "documents": "\\n".join(f"- {s}" for s in session.all_sources),
    "question": question,
})

intent_result

### 4b. Metadata extraction

What filters would we apply to Chroma for this question?

In [ ]:
from simple_rag_chatbot.filters import format_available_metadata

metadata_result = session.chains["metadata_chain"].invoke({
    "history": "No previous conversation.",
    "documents": "\\n".join(f"- {s}" for s in session.all_sources),
    "metadata": format_available_metadata(session.available_metadata),
    "question": question,
})

metadata_result

### 4c. Multi-vector retrieval + Reciprocal Rank Fusion

Embed the question, query the content / summary / entity vector spaces separately, and fuse them.

In [ ]:
from simple_rag_chatbot.vectorstore import multivector_retrieve

query_embedding = session.embedder.encode([question]).tolist()

fused, content_r, summary_r, metadata_r = multivector_retrieve(
    session.collection,
    session.summary_vector_collection,
    session.metadata_vector_collection,
    query_embedding,
    candidate_k=config.candidate_k,
)

print(
    f"Content candidates: {len(content_r)} | "
    f"Summary candidates: {len(summary_r)} | "
    f"Metadata candidates: {len(metadata_r)} | "
    f"Fused: {len(fused)}"
)

for item in fused[:5]:
    print(
        "-", item["metadata"].get("source_file"),
        "|", item["document"][:80].replace("\\n", " ")
    )

### 4d. Reranking

The CrossEncoder reranker scores each (question, chunk) pair directly, which is usually more accurate than embedding similarity alone.

In [ ]:
pairs = [(question, item["document"]) for item in fused]
scores = session.reranker.predict(pairs)

ranked = sorted(
    zip(scores, fused),
    key=lambda x: x[0],
    reverse=True,
)

for score, item in ranked[:config.top_k]:
    print(
        f"{score:.3f} | "
        f"{item['metadata'].get('source_file')} | "
        f"{item['document'][:70].replace(chr(10), ' ')}"
    )

## 5. The full LangGraph workflow

Now put it together. `build_langgraph_rag` wires the stages into one graph:

`START → intent → metadata → rewrite → retrieve → rerank → generate → END`

Other routes include direct response, out-of-scope, document-list, and document-filter handling.

In [ ]:
from IPython.display import Image
from simple_rag_chatbot.graph import build_langgraph_rag

rag_graph = build_langgraph_rag(
    intent_chain=session.chains["intent_chain"],
    metadata_chain=session.chains["metadata_chain"],
    query_rewrite_chain=session.chains["query_rewrite_chain"],
    rag_chain=session.chains["rag_chain"],
    collection=session.collection,
    embedder=session.embedder,
    reranker=session.reranker,
    args=session.config,
    all_sources=session.all_sources,
    available_metadata=session.available_metadata,
    summary_vector_collection=session.summary_vector_collection,
    metadata_vector_collection=session.metadata_vector_collection,
    active_filter=session.active_filter,
)

Image(rag_graph.get_graph().draw_mermaid_png())

In [ ]:
from simple_rag_chatbot.graph import build_langgraph_rag
from simple_rag_chatbot.history import format_history, add_to_history
from simple_rag_chatbot.filters import format_available_metadata

def ask(question, session):
    rag_graph = build_langgraph_rag(
        intent_chain=session.chains["intent_chain"],
        metadata_chain=session.chains["metadata_chain"],
        query_rewrite_chain=session.chains["query_rewrite_chain"],
        rag_chain=session.chains["rag_chain"],
        collection=session.collection,
        embedder=session.embedder,
        reranker=session.reranker,
        args=session.config,
        all_sources=session.all_sources,
        available_metadata=session.available_metadata,
        summary_vector_collection=session.summary_vector_collection,
        metadata_vector_collection=session.metadata_vector_collection,
        active_filter=session.active_filter,
    )

    result = rag_graph.invoke({
        "question": question,
        "history_text": format_history(session.conversation_history),
        "document_list_text": "\\n".join(f"- {s}" for s in session.all_sources),
        "metadata_text": format_available_metadata(session.available_metadata),
    })

    add_to_history(
        session.conversation_history,
        question,
        result.get("answer", ""),
        session.config.history_turns,
    )

    return result

In [ ]:
result = ask("How is employee performance evaluated?", session)
print("Answer:", result["answer"])
print("Sources:", result.get("sources"))

In [ ]:
result.keys()

## Chat in Loop

In [ ]:
from simple_rag_chatbot.cli import chat_loop

chat_loop(session)